# Question.1 Option
Historical volatility is calculated from past observed market data and reflects the realized variability of asset prices over time.

Implied volatility is derived from option prices and represents the market’s expectation of future volatility, as the value that reproduces the observed option premium in a pricing model such as Black-Scholes.

In the provided workbook, the “Option” sheet uses volatility as a model input (Vol = 0.3), which behaves as an implied or assumed volatility. In contrast, the “VaR Calculation” sheet is based on historical market rates and uses daily shifts (1d shift) derived from past data, which corresponds to historical volatility.


# Question.2 VaR
Value at Risk (VaR) is a risk measure that estimates the potential loss of a portfolio over a specified time horizon at a given confidence level. For example, a 1-day 99% VaR represents the loss that is not expected to be exceeded with 99% confidence over one day.

Common methods for calculating VaR include:
- Historical Simulation, which uses past market data to generate PnL scenarios
- Variance-Covariance (Parametric VaR), which assumes a distribution for returns
- Monte Carlo Simulation, which relies on simulated market scenarios

In the provided workbook (Sheet: “VaR Calculation”), the VaR is computed using the historical simulation approach, where daily shifts are derived from historical FX rates and applied to the portfolio to generate PnL scenarios. The VaR is then obtained from the lower tail of the PnL distribution.


# Question 3 Option*

Client implemented Black&Scholes model for vanilla option pricing in their excel spreadsheet based on the formulas in the screenshots, (assuming zero dividends).
- Implement the model** in python code and validate/test the model for Call and Put options.
- test for in the money, at-the-money, out-of-money with attention to unit and end-to-end testing 
- **Feel free to choose any method for model implementation, either with forward price or with spot price or with put-call parity


In [48]:
import math


def norm_cdf(x: float) -> float:
    """
    Standard normal cumulative distribution function.
    """
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def bs_d1_d2(S: float, K: float, r: float, T: float, sigma: float):
    """
    Calculate d1 and d2 for Black-Scholes (no dividends).
    """

    if S <= 0 or K <= 0 or T <= 0 or sigma <= 0:
        raise ValueError("S, K, T, and sigma must be positive.")

    vol_sqrt_t = sigma * math.sqrt(T)

    # d1: measures relative position of spot vs strike
    d1 = (math.log(S / K) + (r + 0.5 * sigma * sigma) * T) / vol_sqrt_t

    # d2: adjusted threshold at expiry
    d2 = d1 - vol_sqrt_t

    return d1, d2


def bs_price(S: float, K: float, r: float, T: float, sigma: float, option_type="call"):
    """
    Black-Scholes price for European call/put (no dividends).
    """

    d1, d2 = bs_d1_d2(S, K, r, T, sigma)

    disc_r = math.exp(-r * T)

    if option_type.lower() == "call":
        # Call formula (no dividend)
        return S * norm_cdf(d1) - K * disc_r * norm_cdf(d2)

    elif option_type.lower() == "put":
        # Put formula (no dividend)
        return K * disc_r * norm_cdf(-d2) - S * norm_cdf(-d1)

    else:
        raise ValueError("option_type must be 'call' or 'put'.")

In [49]:
# =========================
# Validation against Excel ("Option" sheet)
# =========================

# Input parameters taken from the Excel "Option" sheet.
# These correspond to the base case used in the workbook.
S = 19.0       # Spot price
K = 17.0       # Strike price
T = 0.460  # Time to maturity (in years)
r = 0.005      # Risk-free interest rate
sigma = 0.3    # Volatility

# Compute option prices using the implemented Black-Scholes model.
call = bs_price(S, K, r, T, sigma, "call")
put = bs_price(S, K, r, T, sigma, "put")

# Display results for manual inspection.
print("Call price (Python):", call)
print("Put price (Python):", put)

# Expected values from Excel "Option" sheet.
# These values are used as reference for validation.
excel_call = 2.70
excel_put = 0.66


# Print comparison
print("=== Validation against Excel ===")
print(f"Call (Python): {call:.6f}")
print(f"Call (Excel) : {excel_call:.6f}")
print(f"Difference   : {call - excel_call:.6f}")
print()

print(f"Put (Python): {put:.6f}")
print(f"Put (Excel) : {excel_put:.6f}")
print(f"Difference  : {put - excel_put:.6f}")

Call price (Python): 2.696564799408952
Put price (Python): 0.6575097299555992
=== Validation against Excel ===
Call (Python): 2.696565
Call (Excel) : 2.700000
Difference   : -0.003435

Put (Python): 0.657510
Put (Excel) : 0.660000
Difference  : -0.002490


In [50]:
# Test cases for call options using the same spot price.
cases = [
    ("ITM", 15.0),  # Strike below spot: call is in the money
    ("ATM", S),  # Strike approximately equal to spot
    ("OTM", 23.0),  # Strike above spot: call is out of the money
]

for label, test_strike in cases:
    call_test = bs_price(S, test_strike, r, T, sigma, "call")
    put_test = bs_price(S, test_strike, r, T, sigma, "put")

    print(label)
    print("Call:", call_test)
    print("Put:", put_test)

    # Option prices should never be negative.
    assert call_test >= 0
    assert put_test >= 0

ITM
Call: 4.237113961444727
Put: 0.2026536060447084
ATM
Call: 1.55978012575879
Put: 1.516130342252099
OTM
Call: 0.40417330605046065
Put: 4.351334094437101


In [51]:
# =========================
# Test cases: ITM / ATM / OTM with expected behaviour
# =========================

test_cases = [
    ("ITM", 15.0, "Call option should have higher value due to intrinsic value."),
    ("ATM", S, "Call and put values should be moderate since spot is close to strike."),
    ("OTM", 23.0, "Call option should have lower value because strike is above spot."),
]

print("=== ITM / ATM / OTM Test Cases ===")

for label, test_strike, expectation in test_cases:
    call_test = bs_price(S=S, K=test_strike, r=r, T=T, sigma=sigma, option_type="call")
    put_test = bs_price(S=S, K=test_strike, r=r, T=T, sigma=sigma, option_type="put")

    print(f"\nScenario: {label}")
    print(f"Spot Price : {S:.2f}")
    print(f"Strike     : {test_strike:.2f}")
    print(f"Call Price : {call_test:.6f}")
    print(f"Put Price  : {put_test:.6f}")
    print(f"Expected   : {expectation}")

=== ITM / ATM / OTM Test Cases ===

Scenario: ITM
Spot Price : 19.00
Strike     : 15.00
Call Price : 4.237114
Put Price  : 0.202654
Expected   : Call option should have higher value due to intrinsic value.

Scenario: ATM
Spot Price : 19.00
Strike     : 19.00
Call Price : 1.559780
Put Price  : 1.516130
Expected   : Call and put values should be moderate since spot is close to strike.

Scenario: OTM
Spot Price : 19.00
Strike     : 23.00
Call Price : 0.404173
Put Price  : 4.351334
Expected   : Call option should have lower value because strike is above spot.


The Black-Scholes model was implemented in Python for European vanilla call and put options assuming zero dividends. The implementation was validated against the values in the “Option” sheet and tested under ITM, ATM, and OTM scenarios. The results show the expected pricing behavior across different strike levels.

# Question 4 VaR*

- Bank has FX portfolio consisting of two currencies. Risk manager implemented historical VaR methodology as defined in internal policies (1 day with .99 confidence level) for the FX portfolio in excel, assuming no correlation between these currencies.
- implement the VaR calculation in python to revert one day VaR value

One year of historical data (market rate) for the currencies and current/spot portfolio value is known.


ANSWER : The VaR was calculated using a historical simulation approach based on one year of FX market data. Daily logarithmic shifts were derived from historical market rates and converted into standard returns, which were then applied to the spot portfolio values to generate PnL vectors for each currency. In this context, the position corresponds to the spot portfolio value, and PnL is calculated as position multiplied by the market return. The total portfolio PnL was obtained by summing the PnL of both currencies, assuming no correlation. The resulting PnL distribution was sorted, and the 1% lower percentile was estimated using linear interpolation to obtain the 1-day 99% VaR.

In [52]:
# =========================
# Data fetching and preprocessing for market rates
# =========================
import pandas as pd
file_path = 'C:\\Users\\ETEPEBAS\\Documents\\interview_assigment\\TRM_Engineering_Data.csv'
lines  = []
with open(file_path, 'r', encoding='utf-8-sig') as f:
    for _ in range(2):
        lines.append(f.readline())
start_row = len(lines)
top_header = [part.strip() for part in lines[0].split(';')]
base_header = [part.strip() for part in lines[1].split(';')]

column_names = []
for i, (group, name) in enumerate(zip(top_header, base_header)):
    if i == 0:
        column_names.append('row_id')
    elif name and group and group != name:
        column_names.append(f'{name} {group}')
    else:
        column_names.append(name or group or f'column_{i}')

df_market_rate = pd.read_csv(
    file_path,
    sep=';',
    skiprows=start_row,
    header=None,
    names=column_names,
    index_col=0,
    decimal=',',
    encoding='utf-8-sig',
)
df_market_rate.head()

,date,Portfolio,market rate ccy-1,market rate ccy-2,1d shift ccy-1,1d shift ccy-2,Pnl Vector ccy-1,Pnl Vector ccy-2,Total PnL FXTEST Total PnL vector
row_id,,,,,,,,,
1,14-11-2019,FXTEST,1.168443,0.886564,0.002115,0.002527,323.756200,242.289821,566.046021
2,13-11-2019,FXTEST,1.165977,0.884330,-0.000047,0.000973,-7.139733,93.279679,86.139946
3,12-11-2019,FXTEST,1.166031,0.883470,-0.000746,0.006759,-114.241063,648.087332,533.846269
4,11-11-2019,FXTEST,1.166902,0.877539,0.005321,-0.000136,814.575462,-13.061579,801.513883
5,8-11-2019,FXTEST,1.160726,0.877659,0.000012,0.001584,1.776895,151.927424,153.704319


In [53]:
import numpy as np

# Extract total PnL
total_pnl = df_market_rate["Total PnL FXTEST Total PnL vector"].dropna()

# Sort values (ascending)
sorted_pnl = np.sort(total_pnl)

# Number of observations
n = len(sorted_pnl)

# 1% percentile index
percentile_index = 0.01 * n

# Adjust for 0-based indexing
lower_index = int(np.floor(percentile_index)) - 1 # -1 for Adjust for 0-based indexing
upper_index = int(np.ceil(percentile_index)) - 1 # -1 for Adjust for 0-based indexing

# Interpolation weight
weight = percentile_index - np.floor(percentile_index)

# VaR calculation
var = (
    (1 - weight) * sorted_pnl[lower_index] +
    weight * sorted_pnl[upper_index]
)

print(f"VaR (Python): {var:.2f}")
print("VaR (Excel):   -13.572,73")

VaR (Python): -13595.86
VaR (Excel):   -13.572,73


The difference between Python and Excel results is due to different percentile and interpolation approaches, with Excel applying rounding while Python uses exact interpolation.